# DEH 30-Day PySpark Challenge
## Day 3 — Schema: Infer vs Define

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

## Step 1 — Install PySpark

In [ ]:
!pip install pyspark --quiet

## Step 2 — AWS Credentials & Bucket

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

## Step 3 — Create SparkSession

In [ ]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType,
    LongType, BooleanType, TimestampType
)

spark = (
    SparkSession
    .builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

sc = spark.sparkContext
print(f"Spark version : {spark.version}")

## Step 4 — Default Read (No Schema)

Without any options, every column comes in as a string.

In [ ]:
# Default read — everything is StringType
orders_default = spark.read.csv(
    f"s3a://{BUCKET}/data/orders.csv",
    header=True
)

print("--- Default schema (no inferSchema) ---")
orders_default.printSchema()

## Step 5 — inferSchema=True

Spark scans the data and guesses types. Reads the file twice.

In [ ]:
# inferSchema — Spark guesses types
orders_infer = spark.read.csv(
    f"s3a://{BUCKET}/data/orders.csv",
    header=True,
    inferSchema=True
)

print("--- Inferred schema ---")
orders_infer.printSchema()

## Step 6 — Explicit Schema with StructType

Define the schema yourself — faster, safer, production-ready.

In [ ]:
# Define the schema explicitly
orders_schema = StructType([
    StructField("order_id",       StringType(),   True),
    StructField("customer_id",    StringType(),   True),
    StructField("product_id",     StringType(),   True),
    StructField("order_date",     DateType(),     True),
    StructField("quantity",       IntegerType(),  True),
    StructField("unit_price",     DoubleType(),   True),
    StructField("discount_pct",   IntegerType(),  True),
    StructField("status",         StringType(),   True),
    StructField("payment_method", StringType(),   True),
    StructField("region",         StringType(),   True)
])

# Read with explicit schema
orders_df = spark.read.csv(
    f"s3a://{BUCKET}/data/orders.csv",
    header=True,
    schema=orders_schema
)

print("--- Explicit schema ---")
orders_df.printSchema()
orders_df.show(3)

## Step 7 — nullable=True vs nullable=False

In [ ]:
# nullable=False marks a column as required
# Useful for primary keys and required fields
strict_schema = StructType([
    StructField("order_id",    StringType(),  False),  # required — primary key
    StructField("customer_id", StringType(),  False),  # required
    StructField("unit_price",  DoubleType(),  True),   # optional — can be null
    StructField("status",      StringType(),  True)
])

# Note: nullable=False is metadata only for CSV reads
# It is enforced properly in Delta Lake and Iceberg tables
print("nullable=False marks a column as a required field")
print("This is metadata — documents intent and used by downstream operations")

---
## Assignment — Day 3

---

### Task 1
Read `products.csv` from S3 with no options. Print the schema.  
Then read it again with `inferSchema=True`. Print that schema.  
What columns changed type?

In [ ]:
# Task 1 — Write your code here


### Task 2
Define an explicit `StructType` schema for `products.csv` with these columns:
- `product_id` (String), `product_name` (String), `category` (String), `sub_category` (String)
- `unit_price` (Double), `cost_price` (Double), `supplier` (String), `stock_quantity` (Integer)

Read the file using your schema and print the schema to verify.

In [ ]:
# Task 2 — Write your code here


### Task 3
Using the products DataFrame with your explicit schema, calculate the profit margin:  
`(unit_price - cost_price) / unit_price * 100`

Show `product_name`, `unit_price`, `cost_price`, and your calculated `profit_margin` column.  
Could you have done this calculation without an explicit schema?

In [ ]:
# Task 3 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*